# Kaggle Legal Retrieval Corpus Preparation

## Purpose

This notebook prepares a retrieval corpus from the **Kaggle-normalized dataset** (from notebook 01).

**Baseline pipeline:**
This notebook builds the primary baseline corpus for Notebooks 03-06 (retrieval, reranking, generation, evaluation).

**What this notebook does:**
- Loads the **Kaggle-normalized** QA dataset (`kaggle_normalized.csv`)
- Creates structured retrieval documents combining question, answer, source, and category fields
- Applies text cleaning preserving Turkish characters
- Chunks documents by character length with overlap for better retrieval coverage
- Generates metadata (doc_id, chunk_id, text lengths, etc.)
- Performs quality checks on the corpus
- Saves retrieval corpus in CSV and JSONL formats

**What this notebook does NOT do:**
- Build embeddings or vector indices
- Implement BM25 or keyword search
- Evaluate retrieval performance
- Use sentence-transformers or deep learning models

**Optional:** At the end, there is an optional section to generate a merged corpus (Kaggle + HF) for later comparison.

**Note:** Keeping the corpus separate from the merged dataset ensures metadata quality and allows for controlled ablation studies.

## 1. Environment Setup

In [ ]:
import os
import json
import uuid
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional, Tuple

## 2. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted")

## 3. Configuration and Paths

In [ ]:
# ===================== PROJECT CONFIGURATION =====================
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"

# ===================== DATASET MODE =====================
# Set to "kaggle_only" for baseline corpus (primary)
# Set to "merged" for combined Kaggle+HF corpus (optional comparison)
DATASET_MODE = "kaggle_only"

# ===================== INPUT PATHS =====================
# Produced by Notebook 01 — do NOT modify these files
KAGGLE_DATA_PATH = f"{PROJECT_ROOT}/data/processed/kaggle_normalized.csv"
HF_DATA_PATH = f"{PROJECT_ROOT}/data/processed/hf_normalized.csv"
MERGED_DATA_PATH = f"{PROJECT_ROOT}/data/processed/merged_legal_qa.csv"

# ===================== OUTPUT CONFIGURATION =====================
RETRIEVAL_OUTPUT_DIR = f"{PROJECT_ROOT}/data/retrieval"

# Select input and output based on dataset mode
if DATASET_MODE == "kaggle_only":
    INPUT_DATA_PATH = KAGGLE_DATA_PATH
    CORPUS_OUTPUT_NAME = "kaggle_retrieval_corpus.csv"
    MODE_LABEL = "Kaggle-only (baseline)"
elif DATASET_MODE == "merged":
    INPUT_DATA_PATH = MERGED_DATA_PATH
    CORPUS_OUTPUT_NAME = "merged_retrieval_corpus.csv"
    MODE_LABEL = "Merged (Kaggle + HF)"
else:
    raise ValueError(f"Unknown DATASET_MODE: {DATASET_MODE}")

# ===================== CHUNKING CONFIGURATION =====================
CHUNK_MAX_CHARS = 800  # Character limit per chunk
CHUNK_OVERLAP = 100    # Character overlap between chunks

# ===================== CREATE OUTPUT DIRECTORY =====================
from pathlib import Path
Path(RETRIEVAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ===================== PRINT CONFIGURATION =====================
print("="*80)
print("NOTEBOOK 02 — RETRIEVAL CORPUS PREPARATION")
print("="*80)
print(f"\nDataset Mode: {MODE_LABEL}")
print(f"Input data: {INPUT_DATA_PATH}")
print(f"Output directory: {RETRIEVAL_OUTPUT_DIR}")
print(f"Corpus output: {CORPUS_OUTPUT_NAME}")
print(f"\nChunking settings:")
print(f"  - Max chars per chunk: {CHUNK_MAX_CHARS}")
print(f"  - Overlap: {CHUNK_OVERLAP}")
print(f"\nNote: This is the PRIMARY BASELINE corpus.")
print(f"      Notebooks 03-06 will use: {CORPUS_OUTPUT_NAME}")
print("="*80)

## 4. Load Dataset

In [ ]:
# Load dataset based on DATASET_MODE
print(f"Loading {MODE_LABEL} dataset...")
df = pd.read_csv(INPUT_DATA_PATH)
print(f"✓ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")

# Store in a clear variable name for subsequent cells
print(f"\n✓ Dataset ready for corpus building")

## 5. Dataset Inspection

In [ ]:
print("="*80)
print(f"DATASET OVERVIEW — {MODE_LABEL}")
print("="*80)

print(f"\nShape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum())

print(f"\nDataset composition:")
if 'dataset_name' in df.columns:
    print(df["dataset_name"].value_counts())
else:
    print("  (No dataset_name column)")

print(f"\nFirst 2 sample rows:")
print(df.head(2))

In [ ]:
# Check source and category coverage
print(f"\nSource coverage:")
print(f"  - Non-null: {df['source'].notna().sum()}")
print(f"  - Null: {df['source'].isna().sum()}")
if df['source'].notna().any():
    print(f"  - Unique sources: {df['source'].nunique()}")

print(f"\nCategory coverage:")
print(f"  - Non-null: {df['category'].notna().sum()}")
print(f"  - Null: {df['category'].isna().sum()}")
if df['category'].notna().any():
    print(f"  - Unique categories: {df['category'].nunique()}")
    print(f"\n  Top categories:")
    print(df['category'].value_counts().head(10))

print(f"\n📌 NOTE: This {MODE_LABEL} dataset forms the basis for Notebooks 03-06.")

In [ ]:
def build_retrieval_text(row: pd.Series) -> str:
    """
    Build full retrieval text from row fields.
    Includes source, category, question, and answer when available.
    """
    parts = []
    
    if pd.notna(row.get('source')):
        parts.append(f"Source: {row['source']}")
    
    if pd.notna(row.get('category')):
        parts.append(f"Category: {row['category']}")
    
    if pd.notna(row.get('question')):
        parts.append(f"Question: {row['question']}")
    
    if pd.notna(row.get('answer')):
        parts.append(f"Answer: {row['answer']}")
    
    return "\n".join(parts)


def build_candidate_text(row: pd.Series) -> str:
    """
    Build shorter candidate text from question and answer.
    """
    parts = []
    
    if pd.notna(row.get('question')):
        parts.append(str(row['question']))
    
    if pd.notna(row.get('answer')):
        parts.append(str(row['answer']))
    
    return "\n".join(parts)


# Build retrieval text and candidate text for each row
df['retrieval_text'] = df.apply(build_retrieval_text, axis=1)
df['candidate_text'] = df.apply(build_candidate_text, axis=1)

print(f"✓ Built retrieval text fields")
print(f"\nExample retrieval text:\n")
print(df['retrieval_text'].iloc[0][:300])

In [ ]:
# Build retrieval text and candidate text for each row
df['retrieval_text'] = df.apply(build_retrieval_text, axis=1)
df['candidate_text'] = df.apply(build_candidate_text, axis=1)

print(f"✓ Built retrieval text fields")
print(f"\nExample retrieval text:\n")
print(df['retrieval_text'].iloc[0][:300])

In [ ]:
def clean_retrieval_text(text: str) -> str:
    """
    Clean retrieval text while preserving Turkish characters.
    """
    if not isinstance(text, str):
        return ""
    
    # Remove extra whitespace
    text = " ".join(text.split())
    
    # Remove special characters but preserve Turkish characters and punctuation
    # Keep: Turkish letters (ç, ğ, ı, ö, ş, ü, Ç, Ğ, İ, Ö, Ş, Ü)
    # Keep: English letters, numbers, spaces, punctuation
    # Remove: unusual control characters
    
    return text.strip()


# Apply cleaning
df['retrieval_text_clean'] = df['retrieval_text'].apply(clean_retrieval_text)
df['candidate_text_clean'] = df['candidate_text'].apply(clean_retrieval_text)

print(f"✓ Cleaned retrieval text fields")
print(f"\nExample cleaned retrieval text:\n")
print(df['retrieval_text_clean'].iloc[0][:300])

## 8. Text Chunking Strategy

In [ ]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100) -> List[str]:
    """
    Chunk text by character length with overlap.
    Preserves Turkish characters and avoids empty chunks.
    
    Args:
        text: Text to chunk
        max_chars: Maximum characters per chunk
        overlap: Character overlap between consecutive chunks
    
    Returns:
        List of chunks (strings)
    """
    if not isinstance(text, str) or len(text) == 0:
        return []
    
    # If text is shorter than max_chars, keep as single chunk
    if len(text) <= max_chars:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        # Calculate end position
        end = start + max_chars
        
        # If this is the last chunk, include remainder
        if end >= len(text):
            chunk = text[start:]
            if chunk.strip():  # Avoid empty chunks
                chunks.append(chunk)
            break
        
        # Try to break at a word boundary (space)
        # Look for last space within chunk
        last_space = text.rfind(' ', start, end)
        if last_space > start:
            end = last_space
        
        chunk = text[start:end].strip()
        if chunk:  # Avoid empty chunks
            chunks.append(chunk)
        
        # Move start position with overlap
        start = end - overlap if end > overlap else end
    
    return chunks


# Test chunking function
test_text = df['retrieval_text_clean'].iloc[0]
test_chunks = chunk_text(test_text, max_chars=CHUNK_MAX_CHARS, overlap=CHUNK_OVERLAP)
print(f"✓ Chunking function ready")
print(f"\nExample: text length={len(test_text)}, chunks={len(test_chunks)}")
if test_chunks:
    print(f"  - Chunk 1 length: {len(test_chunks[0])}")
    if len(test_chunks) > 1:
        print(f"  - Chunk 2 length: {len(test_chunks[1])}")

## 9. Create Retrieval Corpus with Chunks

In [ ]:
# Generate corpus with chunks
corpus_rows = []

for original_idx, row in df.iterrows():
    # Get text to chunk
    text_to_chunk = row['retrieval_text_clean']
    
    # Generate unique doc_id from UUID
    doc_id = f"doc_{original_idx:05d}_{str(uuid.uuid4())[:8]}"
    
    # Chunk the text
    chunks = chunk_text(text_to_chunk, max_chars=CHUNK_MAX_CHARS, overlap=CHUNK_OVERLAP)
    
    # Create corpus row for each chunk
    for chunk_idx, chunk_text_val in enumerate(chunks):
        chunk_id = f"{doc_id}_chunk_{chunk_idx}"
        
        corpus_rows.append({
            'doc_id': doc_id,
            'chunk_id': chunk_id,
            'chunk_index': chunk_idx,
            'original_row_index': original_idx,
            'question': row['question'],
            'answer': row['answer'],
            'source': row['source'],
            'category': row['category'],
            'dataset_name': row.get('dataset_name', 'kaggle'),
            'split': row.get('split', 'train'),
            'quality_score': row.get('quality_score', None),
            'full_text': text_to_chunk,
            'chunk_text': chunk_text_val,
            'text_length': len(text_to_chunk),
            'chunk_length': len(chunk_text_val),
            'n_chunks': len(chunks),
        })

# Create retrieval corpus DataFrame
df_corpus = pd.DataFrame(corpus_rows)
print(f"✓ Created retrieval corpus ({MODE_LABEL})")
print(f"  - Original documents: {df.shape[0]}")
print(f"  - Total chunks: {len(df_corpus)}")
print(f"  - Average chunks per document: {len(df_corpus) / df.shape[0]:.2f}")

In [ ]:
# Show corpus sample
print(f"\nCorpus sample (first 3 rows):")
print(df_corpus[['doc_id', 'chunk_index', 'chunk_length', 'source', 'category']].head(3))

print(f"\nExample chunk text:")
print(df_corpus['chunk_text'].iloc[0][:200])

## 10. Corpus Quality Checks

In [ ]:
print("="*80)
print(f"RETRIEVAL CORPUS QUALITY REPORT — {MODE_LABEL}")
print("="*80)

print(f"\nCorpus Statistics:")
print(f"  - Total chunks: {len(df_corpus)}")
print(f"  - Original documents: {df.shape[0]}")
print(f"  - Avg chunks/document: {len(df_corpus) / df.shape[0]:.2f}")

print(f"\nChunk Length Statistics:")
print(f"  - Min: {df_corpus['chunk_length'].min()}")
print(f"  - Max: {df_corpus['chunk_length'].max()}")
print(f"  - Mean: {df_corpus['chunk_length'].mean():.0f}")
print(f"  - Median: {df_corpus['chunk_length'].median():.0f}")
print(f"  - Std: {df_corpus['chunk_length'].std():.0f}")

print(f"\nFull Text Length Statistics:")
print(f"  - Min: {df_corpus['text_length'].min()}")
print(f"  - Max: {df_corpus['text_length'].max()}")
print(f"  - Mean: {df_corpus['text_length'].mean():.0f}")
print(f"  - Median: {df_corpus['text_length'].median():.0f}")

In [ ]:
print(f"\nMetadata Coverage:")
print(f"  - Source present: {df_corpus['source'].notna().sum()} / {len(df_corpus)}")
print(f"  - Category present: {df_corpus['category'].notna().sum()} / {len(df_corpus)}")
print(f"  - Quality score present: {df_corpus['quality_score'].notna().sum()} / {len(df_corpus)}")

print(f"\nTop Sources:")
top_sources = df_corpus[df_corpus['source'].notna()]['source'].value_counts().head(5)
if len(top_sources) > 0:
    for source, count in top_sources.items():
        print(f"  - {source}: {count}")
else:
    print("  (No sources available)")

print(f"\nTop Categories:")
top_cats = df_corpus[df_corpus['category'].notna()]['category'].value_counts().head(5)
if len(top_cats) > 0:
    for cat, count in top_cats.items():
        print(f"  - {cat}: {count}")
else:
    print("  (No categories available)")

print(f"\nDataset composition:")
print(df_corpus['dataset_name'].value_counts())

In [ ]:
# Show sample chunks
print("\n" + "="*80)
print("SAMPLE CHUNKS FROM CORPUS")
print("="*80)

for idx in range(min(3, len(df_corpus))):
    row = df_corpus.iloc[idx]
    print(f"\n--- Chunk {idx + 1} ---")
    print(f"doc_id: {row['doc_id']}")
    print(f"chunk_index: {row['chunk_index']} / {row['n_chunks']}")
    print(f"source: {row['source']}")
    print(f"category: {row['category']}")
    print(f"chunk_length: {row['chunk_length']}")
    print(f"\nChunk text:\n{row['chunk_text'][:250]}...")

## 11. Save Retrieval Corpus

In [ ]:
# Reorder columns for clarity
column_order = [
    'doc_id', 'chunk_id', 'chunk_index',
    'original_row_index',
    'question', 'answer',
    'source', 'category',
    'dataset_name', 'split', 'quality_score',
    'full_text', 'chunk_text',
    'text_length', 'chunk_length', 'n_chunks'
]

df_corpus = df_corpus[column_order]

print(f"✓ Finalized corpus DataFrame")
print(f"  Shape: {df_corpus.shape}")
print(f"  Columns: {list(df_corpus.columns)}")

In [ ]:
# Save full corpus as CSV
csv_output = f"{RETRIEVAL_OUTPUT_DIR}/{CORPUS_OUTPUT_NAME}"
df_corpus.to_csv(csv_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {csv_output}")
print(f"  Rows: {len(df_corpus)}")
print(f"  Size: {Path(csv_output).stat().st_size / (1024*1024):.2f} MB")
print(f"\n📌 This corpus will be used by Notebooks 03-06 (retrieval, generation, evaluation)")

In [ ]:
# Save full corpus as JSONL
jsonl_output = f"{RETRIEVAL_OUTPUT_DIR}/retrieval_corpus_full.jsonl"

with open(jsonl_output, 'w', encoding='utf-8') as f:
    for idx, row in df_corpus.iterrows():
        json_record = row.to_dict()
        # Convert NaN to None for JSON serialization
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved: {jsonl_output}")
print(f"  Records: {len(df_corpus)}")
print(f"  Size: {Path(jsonl_output).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Save preview (first 200 rows)
preview_output = f"{RETRIEVAL_OUTPUT_DIR}/retrieval_corpus_preview.csv"
df_corpus.head(200).to_csv(preview_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {preview_output}")
print(f"  Preview rows: {min(200, len(df_corpus))}")

## 12. Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("RETRIEVAL CORPUS PREPARATION COMPLETE")
print("="*80)

print(f"\nCorpus Summary ({MODE_LABEL}):")
print(f"  • Original documents: {df.shape[0]:,}")
print(f"  • Total chunks: {len(df_corpus):,}")
print(f"  • Avg chunks per document: {len(df_corpus) / df.shape[0]:.2f}")
print(f"  • Avg chunk length: {df_corpus['chunk_length'].mean():.0f} characters")

print(f"\nOutput Files (saved to {RETRIEVAL_OUTPUT_DIR}):")
output_files = [
    ("retrieval_corpus_full.csv", "Complete corpus with all chunks"),
    ("retrieval_corpus_full.jsonl", "Same data in JSONL format"),
    ("retrieval_corpus_preview.csv", "First 200 rows for preview"),
]
for fname, desc in output_files:
    print(f"  ✓ {fname}")
    print(f"    {desc}")

print(f"\nImportant Notes:")
print(f"  • This is a FIRST-PASS retrieval corpus from QA data")
print(f"  • Ideal corpus would include separate legal documents (laws, articles, precedents)")
print(f"  • Current corpus is suitable for baseline retrieval experiments")
print(f"  • Future steps should augment with a law/article-based document corpus")

print(f"\nNext Steps:")
print(f"  1. ✓ Dataset preparation (notebook 01)")
print(f"  2. ✓ Retrieval corpus preparation (this notebook)")
print(f"  3. → Extract and prepare legal document corpus (laws, articles)")
print(f"  4. → Build embeddings and create FAISS index")
print(f"  5. → Implement BM25 ranking component")
print(f"  6. → Create retrieval evaluation benchmark")
print(f"  7. → Integrate LLM answer generation")
print(f"  8. → Evaluate full RAG pipeline")